In [2]:
### *S20*

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold

In [5]:
# 1. Load the generated features from S19 (e.g., Matminer or CBFV features)
df_features = pd.read_csv('matminer_magpie_features.csv')
initial_count = df_features.shape[1]
print(f"Initial feature count: {initial_count}")

Initial feature count: 132


In [6]:
# 2. Variance Filter (Drop constant or near-constant features)
# Keep only numeric columns for filtering
numeric_df = df_features.select_dtypes(include=[np.number])

In [7]:
# Threshold = 0.0 means completely constant, slightly higher catches near-constant
selector = VarianceThreshold(threshold=0.01)
selector.fit(numeric_df)

VarianceThreshold(threshold=0.01)

In [8]:
# Get columns that passed the variance test
cols_to_keep = numeric_df.columns[selector.get_support()]
df_var_filtered = numeric_df[cols_to_keep]
var_count = df_var_filtered.shape[1]
print(f"Features after Variance Filter: {var_count} (Dropped {initial_count - var_count})")

Features after Variance Filter: 126 (Dropped 6)


In [9]:
# 3. Correlation Filter (Drop pairs with Pearson r > 0.95)
corr_matrix = df_var_filtered.corr().abs()

In [10]:
# Select upper triangle of correlation matrix to avoid dropping both features in a pair
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

In [11]:
# Find features with correlation greater than 0.95
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
df_final = df_var_filtered.drop(columns=to_drop)

In [12]:
final_count = df_final.shape[1]
print(f"Features after Correlation Filter: {final_count} (Dropped {len(to_drop)})")

Features after Correlation Filter: 96 (Dropped 30)


In [13]:
# 4. Save the filtered dataset
df_final.to_csv('filtered_features_s20.csv', index=False)
print("✅ First round of feature selection complete.")

✅ First round of feature selection complete.
